# PyTorch Tensor Shape Workbook — Unsolved

This is a **write-the-code-yourself** notebook. Each section has:

1. Usage examples in this text block
2. One or more exercise cells
3. An assertion that tells you whether your answer is correct

Replace `...` in each exercise. Predict the shape before running the cell.

Shape notation: `b` = batch, `h` = heads/height, `s` = sequence, `c` = channels/features.

In [ ]:
import torch


torch.manual_seed(42)
print("PyTorch:", torch.__version__)

## 1. Creating tensors

```python
torch.tensor([1, 2, 3])       # values supplied by you; shape (3,)
torch.zeros(2, 3)             # shape (2, 3), filled with 0
torch.ones(2, 3)              # shape (2, 3), filled with 1
torch.full((2, 3), 7)         # shape (2, 3), filled with 7
torch.rand(2, 3)              # uniform random numbers in [0, 1)
torch.randn(2, 3)             # standard-normal random numbers
torch.randint(0, 10, (2, 3))  # integers from 0 through 9
torch.arange(0, 10, 2)        # tensor([0, 2, 4, 6, 8])
torch.linspace(0, 1, 5)       # 5 evenly spaced numbers
torch.eye(3)                  # 3 x 3 identity matrix
torch.zeros_like(x)           # same shape, dtype, device as x
```

In [ ]:
# EXERCISE 1A
# Create a float tensor shaped (2, 3), filled with ones.
x = ...  # YOUR CODE

assert x.shape == (2, 3)
assert x.dtype.is_floating_point
assert torch.all(x == 1)


In [ ]:
# EXERCISE 1B
# Create [0, 3, 6, 9, 12, 15] without typing every value.
x = ...  # YOUR CODE

assert torch.equal(x, torch.tensor([0, 3, 6, 9, 12, 15]))
x

## 2. Shape, dtype, and device

```python
x.shape                    # torch.Size([2, 3, 4])
x.ndim                     # number of dimensions: 3
x.numel()                  # total number of elements: 24
x.dtype                    # e.g. torch.float32
x.device                   # e.g. cpu or cuda:0
x.float()                  # convert to float32
x.long()                   # convert to int64
x.to(dtype=torch.float64)  # explicit dtype conversion
x.to(device)               # move to CPU/GPU/MPS
```

In [ ]:
# EXERCISE 2
# Convert x to float32, without changing its shape.
x = torch.tensor([[1, 2], [3, 4]])
y = ...  # YOUR CODE

assert y.shape == x.shape
assert y.dtype == torch.float32
y

## 3. Indexing and slicing

Integer indexing **removes** a dimension; slicing **keeps** it.

```python
x = torch.randn(2, 3, 4)   # (b, s, c)
x[0]                       # (3, 4)    — removed batch dimension
x[0:1]                     # (1, 3, 4) — kept batch dimension
x[:, 1]                    # (2, 4)
x[:, 1:2]                  # (2, 1, 4)
x[..., -1]                 # (2, 3) — last item of final dimension
x[:, ::2, :]               # every second sequence position
x[x > 0]                   # boolean mask; result is flattened
```

In [ ]:
# EXERCISE 3A
# Select the first batch while KEEPING the batch dimension.
# (2, 3, 4) -> (1, 3, 4)
x = torch.arange(24).reshape(2, 3, 4)
y = ...  # YOUR CODE

assert y.shape == (1, 3, 4)
assert torch.equal(y, x[:1])
y

In [ ]:
# EXERCISE 3B
# Select the final channel from every batch and sequence position.
# (b, s, c) -> (b, s)
x = torch.randn(2, 3, 4)
y = ...  # YOUR CODE

assert y.shape == (2, 3)
assert torch.equal(y, x[:, :, -1])
y

## 4. `reshape`, `view`, `flatten`, `unflatten`

The number of elements must stay constant. Use `-1` to let PyTorch infer one size.

```python
x.reshape(2, 12)          # (2, 3, 4) -> (2, 12)
x.reshape(2, -1)          # PyTorch infers 12
x.view(2, 12)             # similar, but requires compatible memory layout
x.flatten()               # (2, 3, 4) -> (24,)
x.flatten(start_dim=1)    # (2, 3, 4) -> (2, 12)
x.flatten(1, 2)           # combine dimensions 1 through 2
x.unflatten(1, (3, 4))    # (2, 12) -> (2, 3, 4)
```

After `transpose`/`permute`, prefer `reshape`, or call `.contiguous()` before `view`.

In [ ]:
# EXERCISE 4A
# Combine h and s.
# (b, h, s, c) -> (b, h*s, c)
x = torch.randn(2, 3, 4, 5)
y = ...  # YOUR CODE

assert y.shape == (2, 12, 5)
assert y.numel() == x.numel()
y.shape

In [ ]:
# EXERCISE 4B
# Keep b intact and flatten every other dimension.
# (b, h, s, c) -> (b, h*s*c)
x = torch.randn(2, 3, 4, 5)
y = ...  # YOUR CODE

assert y.shape == (2, 60)
y.shape

## 5. `unsqueeze` — insert a dimension

`unsqueeze(dim)` inserts a new dimension of size 1. Negative dimensions count from the right.

```python
x = torch.randn(b, h, s, c)
x.unsqueeze(-1)     # (b, h, s, c) -> (b, h, s, c, 1)
x.unsqueeze(0)      # (b, h, s, c) -> (1, b, h, s, c)
x.unsqueeze(1)      # (b, h, s, c) -> (b, 1, h, s, c)
x[..., None]        # same as x.unsqueeze(-1)
x[:, None, ...]     # same as x.unsqueeze(1)
```

In [ ]:
# EXERCISE 5A — your exact example
# (b, h, s, c) -> (b, h, s, c, 1)
x = torch.randn(2, 3, 4, 5)
y = ...  # YOUR CODE

assert y.shape == (2, 3, 4, 5, 1)
assert torch.equal(y[..., 0], x)
y.shape

In [ ]:
# EXERCISE 5B
# Insert a heads dimension between batch and sequence.
# (b, s, c) -> (b, 1, s, c)
x = torch.randn(2, 4, 5)
y = ...  # YOUR CODE

assert y.shape == (2, 1, 4, 5)
y.shape

## 6. `squeeze` — remove size-1 dimensions

```python
x = torch.randn(1, 3, 1, 5)
x.squeeze()         # (1, 3, 1, 5) -> (3, 5); removes ALL size-1 dims
x.squeeze(0)        # (1, 3, 1, 5) -> (3, 1, 5)
x.squeeze(2)        # (1, 3, 1, 5) -> (1, 3, 5)
```

In model code, `squeeze(dim)` is safer than bare `squeeze()`. A batch of size 1 can otherwise lose its batch dimension. If the named dimension is not size 1, nothing happens.

In [ ]:
# EXERCISE 6
# Remove ONLY the final dimension. Keep the size-1 batch dimension.
# (1, h, s, 1) -> (1, h, s)
x = torch.randn(1, 3, 4, 1)
y = ...  # YOUR CODE

assert y.shape == (1, 3, 4)
y.shape

## 7. `transpose`, `permute`, `movedim`

```python
x.transpose(1, 2)       # swap only dimensions 1 and 2
x.transpose(-2, -1)     # swap the final two dimensions
matrix.T                # transpose a 2-D matrix
x.permute(0, 2, 1, 3)   # specify the complete new dimension order
x.movedim(-1, 1)        # move final dimension to position 1
```

For `permute`, write the old dimension labels and their indices first:

```text
old: (b, h, s, c) = indices (0, 1, 2, 3)
new: (b, s, h, c) = indices (0, 2, 1, 3)
```

In [ ]:
# EXERCISE 7A
# (b, h, s, c) -> (b, s, h, c)
x = torch.randn(2, 3, 4, 5)
y = ...  # YOUR CODE

assert y.shape == (2, 4, 3, 5)
assert y[1, 2, 0, 4] == x[1, 0, 2, 4]
y.shape

In [ ]:
# EXERCISE 7B
# Convert channels-last images to channels-first.
# (b, height, width, channels) -> (b, channels, height, width)
x = torch.randn(8, 32, 32, 3)
y = ...  # YOUR CODE

assert y.shape == (8, 3, 32, 32)
y.shape

## 8. `cat` — join on an existing dimension

`cat` does not create a new axis. All other dimensions must agree.

```python
a = torch.randn(2, 3)
b = torch.randn(2, 3)
torch.cat([a, b], dim=0)  # (2, 3) + (2, 3) -> (4, 3)
torch.cat([a, b], dim=1)  # (2, 3) + (2, 3) -> (2, 6)

# Append sequence tokens:
torch.cat([x1, x2], dim=1)  # (b, s1, c) + (b, s2, c) -> (b, s1+s2, c)
```

In [ ]:
# EXERCISE 8
# Append the second sequence to the first.
# (b, s1, c) + (b, s2, c) -> (b, s1+s2, c)
x1 = torch.randn(2, 3, 5)
x2 = torch.randn(2, 7, 5)
y = ...  # YOUR CODE

assert y.shape == (2, 10, 5)
assert torch.equal(y[:, :3], x1)
assert torch.equal(y[:, 3:], x2)
y.shape

## 9. `stack` — join on a new dimension

Every input must have exactly the same shape.

```python
a = torch.randn(2, 3)
b = torch.randn(2, 3)
torch.stack([a, b], dim=0)   # new front dim: (2, 2, 3)
torch.stack([a, b], dim=1)   # new middle dim: (2, 2, 3)
torch.stack([a, b], dim=-1)  # new final dim: (2, 3, 2)
```

Mental model: `stack([a, b], dim=d)` is similar to unsqueezing each tensor at `d`, then concatenating there.

In [ ]:
# EXERCISE 9A
# Combine three (b, c) tensors into (b, 3, c).
a = torch.randn(2, 5)
b = torch.randn(2, 5)
c = torch.randn(2, 5)
y = ...  # YOUR CODE

assert y.shape == (2, 3, 5)
assert torch.equal(y[:, 0], a)
assert torch.equal(y[:, 1], b)
assert torch.equal(y[:, 2], c)
y.shape

In [ ]:
# EXERCISE 9B — decide whether this needs stack or cat
# Four grayscale images each have shape (height, width).
# Build a batch shaped (4, height, width).
images = [torch.randn(28, 28) for _ in range(4)]
batch = ...  # YOUR CODE

assert batch.shape == (4, 28, 28)
batch.shape

## 10. `split`, `chunk`, `unbind`

```python
torch.split(x, 4, dim=1)          # pieces of size 4
torch.split(x, [2, 3, 7], dim=1)  # pieces with exact requested sizes
torch.chunk(x, 3, dim=1)          # request 3 roughly equal chunks
torch.unbind(x, dim=0)            # one tensor per index; removes dim 0
```

`split` thinks in **piece sizes**. `chunk` thinks in **number of pieces**. `unbind` is the inverse of `stack`.

In [ ]:
# EXERCISE 10A
# Split qkv into q, k, v along the final dimension.
# (b, s, 3*c) -> three tensors, each (b, s, c)
qkv = torch.randn(2, 4, 15)
q, k, v = ...  # YOUR CODE

assert q.shape == k.shape == v.shape == (2, 4, 5)
assert torch.equal(torch.cat([q, k, v], dim=-1), qkv)
q.shape, k.shape, v.shape

In [ ]:
# EXERCISE 10B
# Turn a batch tensor into a tuple of individual examples.
# (b, s, c) -> b tensors shaped (s, c)
x = torch.randn(3, 4, 5)
examples = ...  # YOUR CODE

assert len(examples) == 3
assert all(item.shape == (4, 5) for item in examples)
assert torch.equal(torch.stack(examples), x)
[item.shape for item in examples]

## 11. `expand`, `repeat`, `repeat_interleave`

```python
x = torch.randn(2, 1, 5)
x.expand(2, 4, 5)          # expand size-1 dim; memory-efficient view
x.expand(-1, 4, -1)        # -1 means keep existing size
x.repeat(1, 4, 1)          # tile data; actually copies values

labels = torch.tensor([0, 1, 2])
labels.repeat(2)            # [0, 1, 2, 0, 1, 2]
labels.repeat_interleave(2) # [0, 0, 1, 1, 2, 2]
```

Use `expand` when broadcasted views are enough. Use `repeat` when you truly need tiled copies.

In [ ]:
# EXERCISE 11A
# Give the same c-dimensional vector to every batch and sequence position.
# (c,) -> (b, s, c), using unsqueeze + expand (not repeat).
b, s, c = 2, 4, 5
vector = torch.randn(c)
y = ...  # YOUR CODE

assert y.shape == (b, s, c)
assert torch.equal(y[0, 0], vector)
assert torch.equal(y[1, 3], vector)
y.shape

In [ ]:
# EXERCISE 11B
# [0, 1, 2] -> [0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2]
labels = torch.tensor([0, 1, 2])

expected = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1, 2, 2, 2, 2])
y = ...  # YOUR CODE

assert torch.equal(y, expected)
y

## 12. Broadcasting

PyTorch compares shapes from **right to left**. Two dimensions work together when they are equal or one is `1`. Missing leading dimensions behave like size 1.

```python
x = torch.randn(b, s, c)
bias = torch.randn(c)
x + bias                    # (b, s, c) + (c,) -> (b, s, c)

rows = torch.arange(3)[:, None]  # (3, 1)
cols = torch.arange(4)[None, :]  # (1, 4)
rows + cols                      # (3, 4)
```

In [ ]:
# EXERCISE 12
# Add one learnable-style bias per channel to all batches and positions.
x = torch.zeros(2, 4, 3)
bias = torch.tensor([10.0, 20.0, 30.0])
y = ...  # YOUR CODE

assert y.shape == (2, 4, 3)
assert torch.equal(y[0, 0], bias)
assert torch.equal(y[1, 3], bias)
y

## 13. Reductions and `keepdim`

```python
x.sum()                         # reduce every dimension -> scalar
x.sum(dim=-1)                   # remove final dimension
x.mean(dim=(1, 2))              # reduce multiple dimensions
x.mean(dim=-1, keepdim=True)    # final dimension becomes size 1
x.max(dim=-1).values            # maximum values
x.max(dim=-1).indices           # positions of maxima
x.argmax(dim=-1)                # positions only
```

`keepdim=True` is especially useful when the result needs to broadcast back against the original tensor.

In [ ]:
# EXERCISE 13A
# Mean-pool the sequence but keep a size-1 sequence dimension.
# (b, s, c) -> (b, 1, c)
x = torch.randn(2, 4, 5)
y = ...  # YOUR CODE

assert y.shape == (2, 1, 5)
assert torch.allclose(y[:, 0], x.mean(dim=1))
y.shape

In [ ]:
# EXERCISE 13B
# Normalize each row to mean 0 and standard deviation 1.
# Hint: keep dimensions in both reductions so broadcasting works.
x = torch.randn(8, 16)
y = ...  # YOUR CODE

assert torch.allclose(y.mean(dim=1), torch.zeros(8), atol=1e-6)
assert torch.allclose(y.std(dim=1, unbiased=False), torch.ones(8), atol=1e-5)
y.shape

## 14. Matrix multiplication

For `(..., m, k) @ (..., k, n)`, the output is `(..., m, n)`. Leading dimensions are batch dimensions and can broadcast.

```python
a @ b                              # preferred readable syntax
torch.matmul(a, b)                 # same operation
torch.bmm(a, b)                    # strictly 3-D batched matmul
torch.einsum('bmk,bkn->bmn', a, b) # explicit dimension notation
```

In [ ]:
# EXERCISE 14A
# Batched matrix multiplication:
# (b, m, k) @ (b, k, n) -> (b, m, n)
a = torch.randn(2, 3, 4)
b = torch.randn(2, 4, 5)
y = ...  # YOUR CODE

assert y.shape == (2, 3, 5)
assert torch.allclose(y[0], a[0] @ b[0])
y.shape

In [ ]:
# EXERCISE 14B
# Dot-product each pair of c-dimensional vectors. Use einsum.
# (b, s, c), (b, s, c) -> (b, s)
x = torch.randn(2, 4, 5)
z = torch.randn(2, 4, 5)
y = ...  # YOUR CODE

assert y.shape == (2, 4)
assert torch.allclose(y, (x * z).sum(dim=-1))
y.shape

# Mixed shape challenges

No usage hints in this section. Combine the operations you practiced.

In [ ]:
# CHALLENGE 1 — split into attention heads
# (b, s, h*c) -> (b, h, s, c)
# You will need two shape operations.
b, s, h, c = 2, 6, 4, 8
x = torch.randn(b, s, h * c)
y = ...  # YOUR CODE

assert y.shape == (b, h, s, c)
y.shape

In [ ]:
# CHALLENGE 2 — merge attention heads
# Reverse Challenge 1: (b, h, s, c) -> (b, s, h*c)
# Be mindful that permute can make memory non-contiguous.
b, h, s, c = 2, 4, 6, 8
x = torch.randn(b, h, s, c)
y = ...  # YOUR CODE

assert y.shape == (b, s, h * c)
y.shape

In [ ]:
# CHALLENGE 3 — class token
# Expand one learned token across the batch, then prepend it to the sequence.
# token: (1, 1, c), x: (b, s, c) -> y: (b, s+1, c)
b, s, c = 3, 7, 16
token = torch.randn(1, 1, c)
x = torch.randn(b, s, c)
y = ...  # YOUR CODE

assert y.shape == (b, s + 1, c)
assert torch.equal(y[:, 0], token.expand(b, -1, -1)[:, 0])
assert torch.equal(y[:, 1:], x)
y.shape

In [ ]:
# CHALLENGE 4 — per-head attention scores
# q and k: (b, h, s, c)
# scores: (b, h, s, s), where every query is dotted with every key.
# Hint: transpose the final two dimensions of k before matmul.
b, h, s, c = 2, 4, 6, 8
q = torch.randn(b, h, s, c)
k = torch.randn(b, h, s, c)
scores = ...  # YOUR CODE

assert scores.shape == (b, h, s, s)
assert torch.allclose(scores[0, 0, 1, 2], q[0, 0, 1] @ k[0, 0, 2])
scores.shape

## One-line shape reference

| Goal | Pattern |
|---|---|
| Insert dimension | `x.unsqueeze(dim)` |
| Remove size-1 dimension | `x.squeeze(dim)` |
| Change/group shape | `x.reshape(...)` |
| Combine adjacent dimensions | `x.flatten(start_dim, end_dim)` |
| Swap two dimensions | `x.transpose(dim0, dim1)` |
| Reorder dimensions | `x.permute(...)` |
| Join on existing dimension | `torch.cat(xs, dim=...)` |
| Join on new dimension | `torch.stack(xs, dim=...)` |
| Split by size | `torch.split(x, size, dim=...)` |
| Split into N pieces | `torch.chunk(x, N, dim=...)` |
| Expand without copying | `x.expand(...)` |
| Tile with copies | `x.repeat(...)` |
| Repeat individual elements | `x.repeat_interleave(...)` |
| Preserve reduced dimension | `keepdim=True` |